# Midterm Progress: Supervised Crop Yield Prediction Model Comparison

This notebook builds the **supervised learning** part of the agriculture project. The goal here is to compare three regression models for **crop yield prediction**:

- **Random Forest Regressor**
- **XGBoost Regressor**
- **KNN Regressor**

Why this notebook matters for the project:
- Crop yield is a **labeled target**, so this is a **supervised learning** task.
- Random Forest and XGBoost are **ensemble methods**.
- KNN is included as a **simple non-ensemble baseline**.
- The future unsupervised part of the full project can later add **clustering** and **anomaly detection** for risk profiling.

This notebook focuses only on the midterm contribution: **building, tuning, and comparing supervised models fairly and clearly**.


## What we are doing and why

Agricultural yield depends on many interacting variables such as weather, soil chemistry, terrain, vegetation health, and crop management. These relationships are usually **nonlinear**, which is why comparing multiple models is useful.

### Why these models?

**1. Random Forest**
- Strong for tabular data
- Handles nonlinear relationships well
- More interpretable than many advanced models
- Gives feature importance

**2. XGBoost**
- Often one of the best-performing models on structured data
- Captures complex interactions well
- Includes regularization to reduce overfitting

**3. KNN**
- Simple and intuitive baseline
- Predicts yield based on similar observations
- Useful for comparison, even if it may struggle with high-dimensional data

### Why compare them?
We want to answer:
- Which model gives the **lowest prediction error**?
- Which model gives the **best R2 score**?
- Which model is a better fit for the **yield prediction layer** of the overall hybrid framework?


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from xgboost import XGBRegressor


In [ ]:
file_path = 'Agri_yield_prediction.csv'
df = pd.read_csv(file_path)

print('Shape:', df.shape)
df.head()


## Basic preprocessing

This dataset has:
- many numeric environmental and soil variables
- categorical columns such as crop type, soil type, region, and season
- two date columns: `Planting_Date` and `Harvest_Date`

To keep the notebook easy to understand while still making it stronger:
- we convert the date columns into meaningful numeric features
- we one-hot encode categorical columns
- we impute missing values if needed
- we scale features for KNN because KNN is distance-based


In [ ]:
df['Planting_Date'] = pd.to_datetime(df['Planting_Date'])
df['Harvest_Date'] = pd.to_datetime(df['Harvest_Date'])

df['Planting_Month'] = df['Planting_Date'].dt.month
df['Planting_DayOfYear'] = df['Planting_Date'].dt.dayofyear
df['Harvest_Month'] = df['Harvest_Date'].dt.month
df['Harvest_DayOfYear'] = df['Harvest_Date'].dt.dayofyear
df['Growing_Days'] = (df['Harvest_Date'] - df['Planting_Date']).dt.days

df = df.drop(columns=['Planting_Date', 'Harvest_Date'])

print(df.isnull().sum().sort_values(ascending=False).head(10))
print('
Columns after feature engineering:', len(df.columns))


In [ ]:
target = 'Yield'
X = df.drop(columns=[target])
y = df[target]

categorical_cols = X.select_dtypes(include=['object']).columns.tolist()
numeric_cols = X.select_dtypes(exclude=['object']).columns.tolist()

print('Categorical columns:', categorical_cols)
print('
Number of numeric columns:', len(numeric_cols))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training shape:', X_train.shape)
print('Test shape:', X_test.shape)


## Preprocessing pipelines

We build two preprocessing pipelines:

- **Tree pipeline** for Random Forest and XGBoost
  - impute numeric values
  - impute and one-hot encode categorical values

- **Scaled pipeline** for KNN
  - same preprocessing as above
  - plus standard scaling for numeric features because KNN depends on distances


In [ ]:
tree_numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

tree_categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

tree_preprocessor = ColumnTransformer([
    ('num', tree_numeric_transformer, numeric_cols),
    ('cat', tree_categorical_transformer, categorical_cols)
])

knn_numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

knn_categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

knn_preprocessor = ColumnTransformer([
    ('num', knn_numeric_transformer, numeric_cols),
    ('cat', knn_categorical_transformer, categorical_cols)
])


## Model pipelines

Each model is wrapped in a pipeline so the comparison stays fair.


In [ ]:
rf_pipeline = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('model', RandomForestRegressor(random_state=42, n_jobs=-1))
])

xgb_pipeline = Pipeline([
    ('preprocessor', tree_preprocessor),
    ('model', XGBRegressor(
        objective='reg:squarederror',
        random_state=42,
        n_jobs=-1,
        verbosity=0
    ))
])

knn_pipeline = Pipeline([
    ('preprocessor', knn_preprocessor),
    ('model', KNeighborsRegressor())
])


## Hyperparameter tuning

We do **light but meaningful tuning** for the midterm.
This keeps the notebook understandable while still showing model optimization.

We use `RandomizedSearchCV` so the search is not too expensive.


In [ ]:
rf_param_dist = {
    'model__n_estimators': [150, 250, 350],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}

xgb_param_dist = {
    'model__n_estimators': [150, 250, 350],
    'model__max_depth': [3, 5, 7, 9],
    'model__learning_rate': [0.03, 0.05, 0.1],
    'model__subsample': [0.8, 0.9, 1.0],
    'model__colsample_bytree': [0.8, 0.9, 1.0],
    'model__reg_alpha': [0, 0.1, 1],
    'model__reg_lambda': [1, 3, 5]
}

knn_param_dist = {
    'model__n_neighbors': [3, 5, 7, 9, 11, 15],
    'model__weights': ['uniform', 'distance'],
    'model__p': [1, 2]
}


In [ ]:
rf_search = RandomizedSearchCV(
    rf_pipeline,
    param_distributions=rf_param_dist,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

xgb_search = RandomizedSearchCV(
    xgb_pipeline,
    param_distributions=xgb_param_dist,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

knn_search = RandomizedSearchCV(
    knn_pipeline,
    param_distributions=knn_param_dist,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
    verbose=1
)


In [ ]:
def evaluate_model(name, search_obj, X_train, y_train, X_test, y_test):
    search_obj.fit(X_train, y_train)
    best_model = search_obj.best_estimator_
    preds = best_model.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    rmse = mean_squared_error(y_test, preds, squared=False)
    r2 = r2_score(y_test, preds)

    return {
        'Model': name,
        'Best Params': search_obj.best_params_,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2,
        'Estimator': best_model
    }


In [ ]:
results = []

results.append(evaluate_model('Random Forest', rf_search, X_train, y_train, X_test, y_test))
results.append(evaluate_model('XGBoost', xgb_search, X_train, y_train, X_test, y_test))
results.append(evaluate_model('KNN', knn_search, X_train, y_train, X_test, y_test))

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'Estimator'} for r in results])
results_df = results_df.sort_values(by='RMSE').reset_index(drop=True)
results_df


## Interpreting the results

Use the metrics like this:

- **MAE**: average absolute prediction error
- **RMSE**: penalizes larger errors more strongly
- **R2**: how much variance in yield the model explains

A strong model usually has:
- **low MAE**
- **low RMSE**
- **high R2**

In many tabular agricultural datasets, **XGBoost** often performs best, **Random Forest** is often close and easier to explain, and **KNN** works as a simple baseline.


In [ ]:
best_result = min(results, key=lambda x: x['RMSE'])
print('Best model based on RMSE:', best_result['Model'])
print('
Best parameters:')
print(best_result['Best Params'])


## Optional: Feature importance for the tree-based models

This helps explain **which variables matter most** for yield prediction.
That is useful because the overall project aims to be not only predictive, but also **interpretable and action-oriented**.


In [ ]:
def get_feature_names_from_column_transformer(preprocessor):
    output_features = []
    for name, transformer, cols in preprocessor.transformers_:
        if name == 'remainder':
            continue
        if hasattr(transformer, 'named_steps'):
            if 'onehot' in transformer.named_steps:
                ohe = transformer.named_steps['onehot']
                transformed = ohe.get_feature_names_out(cols)
                output_features.extend(transformed)
            else:
                output_features.extend(cols)
        else:
            output_features.extend(cols)
    return output_features


In [ ]:
for model_name in ['Random Forest', 'XGBoost']:
    row = next(r for r in results if r['Model'] == model_name)
    model_pipeline = row['Estimator']
    preprocessor = model_pipeline.named_steps['preprocessor']
    regressor = model_pipeline.named_steps['model']

    feature_names = get_feature_names_from_column_transformer(preprocessor)
    importances = regressor.feature_importances_

    feat_imp_df = pd.DataFrame({
        'Feature': feature_names,
        'Importance': importances
    }).sort_values(by='Importance', ascending=False).head(15)

    print('
Top 15 important features for', model_name)
    display(feat_imp_df)


## Midterm write-up summary

You can directly use the following idea in your report:

> In the supervised learning component, crop yield is treated as the target variable and predicted using three regression models: Random Forest, XGBoost, and KNN. Random Forest and XGBoost were selected because they are strong ensemble approaches for nonlinear tabular data, while KNN was included as a simpler non-ensemble baseline. The models were trained on environmental, soil, vegetation, terrain, and management features. After preprocessing, feature engineering, and hyperparameter tuning, the models were compared using MAE, RMSE, and R2. This comparison helps identify the most suitable model for the yield prediction layer of the final hybrid framework.

### Why this is supervised and not unsupervised
This notebook is **supervised** because the model learns from a known target variable: **Yield**.

### What will be unsupervised later?
The future risk module can use:
- **Clustering** (such as K-Means or GMM) to identify hidden stress profiles
- **Anomaly detection** (such as Isolation Forest) to flag unusual field conditions

Then the full project can combine:
- predicted yield
- cluster membership
- anomaly score

to assign agricultural cases into **urgent, moderate, and low-risk groups**.
